In [ ]:
import os

import coiled
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from frisky import hijack

from srm.downscaling_utils import interpolate_coarse_to_fine_grid
from srm.qa_flags import (
    DIR_QA_FLAG_CONSTANT_INPUTS,
    calculate_thresholds,
    discover_leaves,
    flag_global_exceedances,
    flag_outliers,
    flag_rsds_above_max,
    get_data,
    parse_tag,
    plot_flags,
    run_flag_loop,
    write_individual_flags,
)
from srm.qaqc import calculate_distortion_flags, sign_flip_mask

os.environ["FRISKY_SUMMARY"] = "off"

# A. Define what data arrays exist to traverse

In [ ]:
# --- Run parameters --------------------------------------------------------
# This regional South-Africa-box run covers three GCMs, each written to its own icechunk store
# (same bucket/branch, named by GCM the same way srm.cache.ArtifactCache names pipeline output
# stores). Looping over GCMS -- rather than hardcoding one, as this notebook used to -- is what
# lets every leaf be compared against ITS OWN GCM's catalog and lineage instead of silently
# reusing whichever GCM happened to be hardcoded.


VARIABLES = ["tas", "tasmax", "tasmin", "pr", "rsds"]  # , "hurs"]

In [ ]:
GCMS = ["CESM2-WACCM"]

# BRANCH = "v0.13.0"
# ROOT_DIR = "s3://us-west-2.opendata.source.coop/carbonplan/srm-downscaling/output/production/"
# STORE_SUBSET_ID = "global"

BRANCH = "pr-638-global"
ROOT_DIR = "s3://carbonplan-srm/scratch/output/qa/"
STORE_SUBSET_ID = "global"

In [ ]:
PLOT_FLAG_MAPS = True
BUCKET = "carbonplan-srm"
PREFIX = "scratch/output/qa-intermediate-flags"

## Set up cluster

In [ ]:
cluster = coiled.Cluster(
    name="srm-qaqc-flags-nrh",
    region="us-west-2",
    n_workers=12,
    worker_vm_types=["m8gn.xlarge"],
    scheduler_vm_types="c8g.xlarge",
    spot_policy="spot_with_fallback",
    use_best_zone=True,
    tags={"Project": "SRM"},
    worker_options={"nthreads": 8},
    environ={"ZARR_ASYNC__CONCURRENCY": "128"},
)

client = hijack(cluster.get_client())
client

In [ ]:
[
    trees,
    tags,
    gcms_np,
    scenarios_np,
    variables_np,
    tags_np,
    methods_np,
    debiased_coarse_flags_np,
] = discover_leaves(gcms=GCMS, branch=BRANCH, root_dir=ROOT_DIR, store_subset_id=STORE_SUBSET_ID)

# B. Generate individual time-varying QA flags

## 1. global exceedances

In [ ]:
%%time
run_flag_loop(
    tags=tags,
    trees=trees,
    bucket=BUCKET,
    prefix=PREFIX,
    flag_name="outside_global_plausible_range",
    compute_flag=lambda da, var: flag_global_exceedances(da=da, var=var),
    write_mode="a",
)

# with plotting ~14.5 mins

## 2. Outliers based on observations

### 2a. Calculate outlier bounds

In [ ]:
# Loading thresholds output from step 1 notebook
store = DIR_QA_FLAG_CONSTANT_INPUTS + "doy_obs_thresholds_global.zarr"
combined = xr.open_zarr(store)

In [ ]:
def _split(ds, suffix):
    names = [v for v in ds.data_vars if v.endswith(suffix)]
    return ds[names].rename({v: v[: -len(suffix)] for v in names})


obs_max = _split(combined, "_max")
obs_min = _split(combined, "_min")
obs_max_std = _split(combined, "_max_std")
obs_min_std = _split(combined, "_min_std")

In [ ]:
[outlier_thresh_low, outlier_thresh_high] = calculate_thresholds(
    obs_max, obs_min, obs_max_std, obs_min_std
)

### 2b. Traverse dataset and save flags

In [ ]:
outlier_thresh_low_annual = outlier_thresh_low.min(dim="dayofyear")
outlier_thresh_high_annual = outlier_thresh_high.max(dim="dayofyear")

In [ ]:
run_flag_loop(
    tags=tags,
    trees=trees,
    flag_name="annual_outlier_flag",
    compute_flag=lambda da, var: flag_outliers(
        da=da,
        outlier_thresh_low=outlier_thresh_low_annual[var],
        outlier_thresh_high=outlier_thresh_high_annual[var],
        timescale="annual",
    ),
    bucket=BUCKET,
    prefix=PREFIX,
    write_mode="a",
)

## 3. rsds-specific latitude check

In [ ]:
def load_rsds_lims(
    key: str = "zonal_doy_max_rsds",
    fpath: str = DIR_QA_FLAG_CONSTANT_INPUTS + "zonal_doy_max_rsds.zarr",
) -> xr.DataArray:
    return xr.open_zarr(fpath, group=key)["data"].load()

In [ ]:
zonal_doy_max_rsds = load_rsds_lims()

In [ ]:
run_flag_loop(
    tags=tags,
    trees=trees,
    flag_name="rsds_max_exceeded",
    compute_flag=lambda da, var: flag_rsds_above_max(da=da, zonal_doy_max_rsds=zonal_doy_max_rsds),
    var_filter="rsds",
    bucket=BUCKET,
    prefix=PREFIX,
    write_mode="a",
)

## 4. temperature inconsistencies

In [ ]:
from srm.qa_flags import flag_tasmax_tas_inconsistency, flag_tasmin_tas_inconsistency

In [ ]:
# This could be modified to fit within the run_flag_loop structure

var_filter = "tas"
plot = True

print(len(tags))
for tag in tags:
    gcm, var, scenario, ens, method = parse_tag(tag)
    if var_filter is not None and var != var_filter:
        continue

    tag_tasmin = f"{gcm}_tasmin_{scenario}_{ens}_{method}"
    tag_tasmax = f"{gcm}_tasmax_{scenario}_{ens}_{method}"
    if (tag_tasmin in tags) and (tag_tasmax in tags):
        print(tag)

        tas = get_data(tag=tag, trees=trees)
        tasmin = get_data(tag=tag_tasmin, trees=trees)
        tasmax = get_data(tag=tag_tasmax, trees=trees)

        flag_tas_tasmax = flag_tasmax_tas_inconsistency(tas, tasmax)
        flag_tas_tasmin = flag_tasmin_tas_inconsistency(tas, tasmin)

        flag_tas = (flag_tas_tasmax + flag_tas_tasmin) > 0
        flag_tasmin = flag_tas_tasmin
        flag_tasmax = flag_tas_tasmax

        write_individual_flags(
            flag_data=flag_tas,
            flag_name="temperature_inconsistency",
            tag=tag,
            write_mode="a",
            bucket=BUCKET,
            prefix=PREFIX,
        )

        write_individual_flags(
            flag_data=flag_tasmax,
            flag_name="temperature_inconsistency",
            tag=tag,
            write_mode="a",
            bucket=BUCKET,
            prefix=PREFIX,
        )

        write_individual_flags(
            flag_data=flag_tasmin,
            flag_name="temperature_inconsistency",
            tag=tag,
            write_mode="a",
            bucket=BUCKET,
            prefix=PREFIX,
        )

        if plot:
            plot_flags(flags=flag_tas, time_varying=True, separate_low_high=False)
            plt.show()
            plt.close()

# C. Generate flags that are constant over time

In [ ]:
from srm.qa_flags import (
    SCENARIO_COMPARISONS as scenario_comparisons,
    TREND_VARIABLE_SETTINGS as VARIABLE_SETTINGS,
    calculate_ensemble_mean_deltas,
)

In [ ]:
plot = True
for key in scenario_comparisons:
    print(key)
    comparison_dict = scenario_comparisons[key]
    scenario1 = comparison_dict["scenario1"]
    scenario2 = comparison_dict["scenario2"]
    scenario1_time_slice = comparison_dict["scenario1_time_slice"]
    scenario2_time_slice = comparison_dict["scenario2_time_slice"]

    for gcm in GCMS:
        for var in VARIABLES:
            print(var)
            # Find tags to use in scenario comparison (all ensemble members for this variable and gcm for the two comparison scenarios)
            tags_scenario1 = tags_np[
                (gcms_np == gcm) * (scenarios_np == scenario1) * (variables_np == var)
            ]
            tags_scenario2 = tags_np[
                (gcms_np == gcm) * (scenarios_np == scenario2) * (variables_np == var)
            ]

            # Calculate distortions
            [delta_raw, delta_raw_pct, delta_ds_coarse, delta_ds_coarse_pct, delta_ds] = (
                calculate_ensemble_mean_deltas(
                    variable=var,
                    tags_scenario1=tags_scenario1,
                    tags_scenario2=tags_scenario2,
                    gcm=gcm,
                    scenario1=scenario1,
                    scenario2=scenario2,
                    scenario1_time_slice=scenario1_time_slice,
                    scenario2_time_slice=scenario2_time_slice,
                    trees=trees,
                )
            )

            distortion_absolute = delta_ds_coarse - delta_raw
            distortion_pct = delta_ds_coarse_pct - delta_raw_pct

            # Flag distortions at coarse scale
            abs_tol = VARIABLE_SETTINGS[var]["abs_tol"]
            pct_tol = VARIABLE_SETTINGS[var]["pct_tol"]
            sign_flip_tol = VARIABLE_SETTINGS[var]["sign_flip"]
            scale = VARIABLE_SETTINGS[var]["scale"]

            trend_distortion_flag = calculate_distortion_flags(
                distortion_absolute=distortion_absolute * scale,
                distortion_pct=distortion_pct,
                tolerance_absolute=abs_tol,
                tolerance_pct=pct_tol,
            )

            flipped_sign_flag = sign_flip_mask(
                delta_ds_coarse * scale, delta_raw * scale, threshold=sign_flip_tol
            )

            # Propagate coarse distortion flags to fine scale
            trend_distortion_flag_fine_frac = interpolate_coarse_to_fine_grid(
                da_coarse_to_regrid=trend_distortion_flag.astype("float32"), da_fine_grid=delta_ds
            )
            trend_distortion_flag_fine = trend_distortion_flag_fine_frac > 0

            flipped_sign_flag_fine_frac = interpolate_coarse_to_fine_grid(
                da_coarse_to_regrid=flipped_sign_flag.astype("float32"), da_fine_grid=delta_ds
            )
            flipped_sign_flag_fine = flipped_sign_flag_fine_frac > 0

            if plot:
                plot_flags(
                    flags=trend_distortion_flag_fine, time_varying=False, separate_low_high=False
                )
                plt.show()
                plt.close()

                plot_flags(
                    flags=flipped_sign_flag_fine, time_varying=False, separate_low_high=False
                )
                plt.show()
                plt.close()

            # Write out flags
            tags_to_flag = np.concat([tags_scenario1, tags_scenario2])
            for tag in tags_to_flag:
                print(tag)
                gcm, var, scenario, ens, method = parse_tag(tag)
                write_individual_flags(
                    flag_data=trend_distortion_flag_fine,
                    flag_name="trend_distortion_" + scenario1 + "_" + scenario2,
                    tag=tag,
                    write_mode="a",
                    bucket=BUCKET,
                    prefix=PREFIX,
                )

                write_individual_flags(
                    flag_data=flipped_sign_flag_fine,
                    flag_name="flipped_sign_" + scenario1 + "_" + scenario2,
                    tag=tag,
                    write_mode="a",
                    bucket=BUCKET,
                    prefix=PREFIX,
                )

In [ ]:
if cluster is not None:
    cluster.shutdown()
else:
    print("no cluster was created (cached run); nothing to shut down")

# D. Read in flags

In [ ]:
from srm.qa_flags import get_intermediate_flags

gcm = "CESM2-WACCM"
var = "rsds"
scenario = "ssp245"
ens = "008"
method = "qdmsd"

tag = f"{gcm}_{var}_{scenario}_{ens}_{method}"
flag_ds = get_intermediate_flags(tag=tag, bucket=BUCKET, prefix=PREFIX)

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2)
flag_frac = flag_ds["outside_global_plausible_range"].mean(dim="time")
flag_frac.where(flag_frac > 0).plot(ax=axes[0, 0])
flag_frac = flag_ds["rsds_max_exceeded"].mean(dim="time")
flag_frac.where(flag_frac > 0).plot(ax=axes[0, 1])
flag_frac = flag_ds["annual_outlier_flag"].mean(dim="time")
flag_frac.where(flag_frac > 0).plot(ax=axes[1, 0])
plt.tight_layout()

In [ ]:
flag_ds["rsds_max_exceeded"].sum(dim="time").plot()
plt.plot([28.5], [-29.6], "xr")